# Configuration and imports

In [0]:
# Run-mode controls (F02.2).
#
# run_mode defaults to "test": every table this notebook touches is
# resolved to an isolated *_test Delta table through src.common.environment.
# "live" resolves the normal development tables and is used only after the
# test-mode evidence gate has passed.
#
# force_reload defaults to false. When true, a snapshot slice that already
# loaded successfully is loaded again, overriding the skip rule.
dbutils.widgets.dropdown("run_mode", "test", ["test", "live"], "Run mode")
dbutils.widgets.dropdown("force_reload", "false", ["false", "true"], "Force reload")

# Source provenance inputs (F05.1). Licence, attribution and retrieval
# time are immutable once the manifest row is stored, so licence and
# attribution have NO defaults: verify them against the current official
# Inside Airbnb data policy (insideairbnb.com/get-the-data) before
# registering. retrieved_at_utc defaults to the first date the file was
# verified in the Volume (the actual download time was not recorded).
dbutils.widgets.text(
    "source_url",
    "https://insideairbnb.com/get-the-data/",
    "Official source landing page URL",
)
dbutils.widgets.text("source_license", "", "Verified licence (e.g. CC-BY-4.0)")
dbutils.widgets.text("source_attribution", "", "Verified attribution text")
dbutils.widgets.text(
    "retrieved_at_utc",
    "2026-11-12T00:00:00Z",
    "Retrieved at (UTC ISO-8601; actual download time if known)",
)

In [0]:
# Safe reruns for the Toronto Airbnb Bronze ingestion, with test/live
# run-mode isolation (F02.2).
#
# Since F05.2 the notebook reads the immutable F04 landing object resolved
# from the source-file manifest (never the root or incoming path), adds
# provenance metadata, atomically replaces only the matching
# city/snapshot slice, and records the run in an ingestion-control table.

from datetime import UTC, datetime

from pyspark.sql import functions as F

from pathlib import Path
import sys

start = Path.cwd().resolve()
ROOT = next(
    (p for p in [start, *start.parents]
     if (p / 'src').is_dir() and (p / 'config/project.yml').is_file()),
    None,
)
if ROOT is None:
    raise RuntimeError('Cannot locate repository root. Open this notebook inside the Databricks Git folder.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print('Repository root:', ROOT)

from src.common.environment import (
    decide_should_run,
    resolve_tables,
    validate_force_reload,
    validate_run_mode,
)
from src.airbnb.ingestion import write_snapshot
from src.common.provenance import build_provenance_columns, build_run_id

# Real Databricks environment values.
CATALOG = "trippilot_dev"
BRONZE_SCHEMA = "bronze"

# Source details for the Toronto Airbnb snapshot.
SOURCE = "insideairbnb"
CITY = "toronto"
SNAPSHOT_DATE = "2026-09-16"
FILE_TYPE = "csv.gz"
# Location of the originally downloaded root file. Since F05.2 this is
# NOT the Bronze input: it is only used by the one-time registration
# cells to stage the file. Bronze reads the manifest landing_path.
RAW_PATH = (
    "/Volumes/trippilot_dev/bronze/source_files/"
    "insideairbnb/toronto/listings.csv.gz"
)

# Run mode and force reload come from the widgets at the top of this
# notebook. The environment module, not the widget, enforces that test
# mode can only resolve isolated *_test tables.
RUN_MODE = validate_run_mode(dbutils.widgets.get("run_mode"))
FORCE_RELOAD = validate_force_reload(dbutils.widgets.get("force_reload"))
TABLES = resolve_tables(RUN_MODE)

# Destination Bronze table.
BRONZE_TABLE = TABLES["bronze"]

# Operational audit table used to prevent duplicate loads.
CONTROL_TABLE = TABLES["bronze_control"]

print(f"Run mode: {RUN_MODE} (force_reload={FORCE_RELOAD})")
print(f"Bronze table: {BRONZE_TABLE}")
print(f"Control table: {CONTROL_TABLE}")

# Check whether this run should proceed

In [0]:
# Ensure the control table exists before reading it.
#
# In test mode this creates the isolated control table on its first run;
# in live mode the table already exists and this is a no-op.
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CONTROL_TABLE} (
      city STRING,
      snapshot_date STRING,
      run_id STRING,
      status STRING,
      row_count LONG,
      started_at_utc STRING,
      completed_at_utc STRING,
      error_message STRING
    ) USING DELTA
""")

# Read prior control records for this exact city and snapshot.
#
# If a previous run already succeeded, we stop before reading or writing
# data. This is the main duplicate-prevention check. force_reload=true
# overrides the skip decision so the slice can be replaced on purpose.

existing_controls = [
    row.asDict()
    for row in spark.table(CONTROL_TABLE)
    .filter(
        (F.col("city") == CITY)
        & (F.col("snapshot_date") == SNAPSHOT_DATE)
    )
    .collect()
]

# Combine the reusable skip rule with the force-reload setting.
should_run = decide_should_run(
    force_reload=FORCE_RELOAD,
    existing_controls=existing_controls,
    city=CITY,
    snapshot_date=SNAPSHOT_DATE,
)

print(f"Existing control records: {len(existing_controls)}")
print(f"Should run ingestion: {should_run}")

# Stop cleanly if this exact city/snapshot has already loaded successfully
# and the user did not ask for a forced replacement.
assert should_run, (
    f"Skipping ingestion: {CITY} / {SNAPSHOT_DATE} already loaded successfully. "
    "Set force_reload=true to replace this slice."
)

# Start the control record

In [0]:
# Create a unique ID for this pipeline execution.
run_id = build_run_id()

# Record when this run started, in UTC.
started_at_utc = datetime.now(UTC).isoformat()

# Insert a "running" record before doing any data work.
# If a later cell fails, this record can be updated to "failed".
spark.sql(f"""
    INSERT INTO {CONTROL_TABLE}
    VALUES (
        '{CITY}',
        '{SNAPSHOT_DATE}',
        '{run_id}',
        'running',
        0,
        '{started_at_utc}',
        NULL,
        NULL
    )
""")

print(f"Started ingestion run: {run_id}")

# Register and resolve the source file (F05.1 / F05.2)

Bronze never parses an ungoverned file. First the exact source file is
registered as a governed source file in the F04 manifest: a staged copy
under `_incoming` is validated and landed immutably by
`src.common.source_acquisition`, and the manifest row is persisted by
`src.common.source_manifest_store` to the table selected by the run mode
(test → `..._test`, live → `trippilot_dev.ops.source_file_manifest`).
When a landed record already exists the registration is skipped and
the record is resolved by stable source identity instead: since F05.2
the Bronze read path below consumes only that immutable `landing_path`,
never the root file or the incoming staging copy. The original root
file is never moved or overwritten.

In [0]:
# F05.2: resolve the landed manifest record for this source identity.
#
# Bronze reads ONLY the immutable F04 landing object recorded in the
# manifest. If a landed record exists this notebook reuses it as-is and
# the registration cells below are skipped. When the record is missing
# the cells below register the file; when it is ambiguous (several
# landed hash versions) or has no landing path the run stops - there
# is NO fallback to the old root Volume path or the incoming staging
# copy.
from src.common.source_manifest_store import ManifestStore

manifest_store = ManifestStore(TABLES["source_manifest"], spark=spark)
manifest_store.ensure_table()

try:
    source_record = manifest_store.read_landed_by_identity(
        SOURCE, CITY, SNAPSHOT_DATE, FILE_TYPE
    )
except ValueError as resolve_error:
    if "records found" in str(resolve_error):
        # Several landed hash versions exist: the operator must decide
        # which source_file_id to process. Never guess.
        raise
    print(resolve_error)
    source_record = None

if source_record is not None:
    print(f"manifest table  : {TABLES['source_manifest']}")
    print(f"source_file_id  : {source_record['source_file_id']}")
    print(f"file hash       : {source_record['file_hash_sha256']}")
    print(f"schema_version  : {source_record['schema_version']}")
    print(f"landing_path    : {source_record['landing_path']}")
    print("Landed record resolved; the registration cells below are skipped.")
else:
    print("No landed record yet; the cells below stage and register the file.")

In [0]:
# F05.1-T01: stage a copy of the raw file under _incoming and inspect
# the exact header of the bytes being registered.
#
# The original root Volume file is never moved or overwritten: only a
# copy is staged, and only the staged copy is registered and landed.
# This registration step runs only when no landed manifest record
# exists for this source identity yet (F05.2).
import shutil

from src.airbnb.source_inspection import inspect_csv_gz_header
from src.common.source_manifest import compute_file_hash_sha256

if source_record is not None:
    print("Landed manifest record already present; registration not needed.")
else:
    SOURCE_ROOT = "/Volumes/trippilot_dev/bronze/source_files"
    INCOMING_DIR = f"{SOURCE_ROOT}/_incoming/{SOURCE}/{CITY}"
    STAGED_PATH = f"{INCOMING_DIR}/listings.csv.gz"

    staged = Path(STAGED_PATH)
    staged.parent.mkdir(parents=True, exist_ok=True)
    if not staged.exists() or compute_file_hash_sha256(
        staged
    ) != compute_file_hash_sha256(RAW_PATH):
        if staged.exists():
            staged.unlink()  # replace a stale staged copy; never a landed file
        shutil.copyfile(RAW_PATH, staged)

    inspection = inspect_csv_gz_header(STAGED_PATH)
    print(f"staged copy     : {STAGED_PATH}")
    print(f"columns         : {inspection['column_count']}")
    print(f"duplicates      : {inspection['duplicate_columns'] or 'none'}")
    print(f"missing/empty   : {inspection['missing_columns'] or 'none'}")
    print(f"schema fingerprint: {inspection['schema_fingerprint']}")
    print(f"schema_version  : {inspection['schema_version']}")

    assert inspection["column_count"] > 0
    assert not inspection["duplicate_columns"], "duplicate header names need review"
    assert not inspection["missing_columns"], "empty header names need review"
    assert compute_file_hash_sha256(STAGED_PATH) == compute_file_hash_sha256(RAW_PATH)
    print("Staged copy hash equals the original raw file hash.")

In [0]:
# F05.1-T02: register the staged file in the manifest and land it
# immutably under the family/city/snapshot hierarchy.
#
# The provenance widgets carry operator-verified values: licence and
# attribution must be confirmed against the current official Inside
# Airbnb data policy before registration - they are immutable in the
# manifest table once stored. The manifest builder computes the file
# hash, byte size, deterministic source_file_id and run_id; the
# acquisition module validates and lands the bytes; the store persists
# every state transition. Runs only when no landed record exists yet.
from src.common.source_acquisition import acquire_source_file

if source_record is not None:
    print("Landed manifest record already present; registration not needed.")
else:
    SOURCE_URL = dbutils.widgets.get("source_url").strip()
    LICENSE = dbutils.widgets.get("source_license").strip()
    ATTRIBUTION = dbutils.widgets.get("source_attribution").strip()
    RETRIEVED_AT_UTC = dbutils.widgets.get("retrieved_at_utc").strip()

    if not SOURCE_URL or not LICENSE or not ATTRIBUTION or not RETRIEVED_AT_UTC:
        raise ValueError(
            "All source provenance widgets must be set before registration: "
            "verify source_url, licence and attribution against the current "
            "official Inside Airbnb policy, and provide the retrieval time. "
            "These values are immutable once the manifest row is stored."
        )

    source_metadata = {
        "source_family": SOURCE,
        "source_entity_id": CITY,
        "source_version": SNAPSHOT_DATE,
        "file_type": FILE_TYPE,
        "source_url": SOURCE_URL,
        "retrieved_at_utc": RETRIEVED_AT_UTC,
        "schema_version": inspection["schema_version"],
        "license": LICENSE,
        "attribution": ATTRIBUTION,
        "source_path": STAGED_PATH,
    }

    source_record = acquire_source_file(
        source_metadata, STAGED_PATH, SOURCE_ROOT, manifest_store=manifest_store
    )

    # Registration evidence: deterministic identity, verified hash,
    # positive byte size, immutable landing path under
    # family/city/snapshot, and the untouched original raw file.
    assert source_record["processing_state"] == "landed"
    assert source_record["file_hash_sha256"] == compute_file_hash_sha256(STAGED_PATH)
    assert source_record["byte_size"] == Path(RAW_PATH).stat().st_size
    assert source_record["byte_size"] > 0
    assert source_record["landing_path"].startswith(
        f"{SOURCE_ROOT}/{SOURCE}/{CITY}/{SNAPSHOT_DATE}/"
    )
    assert compute_file_hash_sha256(RAW_PATH) == source_record["file_hash_sha256"]

    print(f"manifest table  : {TABLES['source_manifest']}")
    print(f"source_file_id  : {source_record['source_file_id']}")
    print(f"landing_path    : {source_record['landing_path']}")
    print(f"byte_size       : {source_record['byte_size']}")
    print("Original raw file unchanged: hash verified.")

In [0]:
# F05.2-T03: validate the landed object before parsing it.
#
# The landed bytes are verified against the manifest (SHA-256 hash and
# byte size), then the header is inspected and compared with the
# schema fingerprint recorded at registration. This blocks
# missing/duplicate columns, empty files, malformed CSV and schema
# drift BEFORE any Spark parsing or table write.
from src.airbnb.source_inspection import inspect_csv_gz_header
from src.common.source_acquisition import verify_landed_record

assert source_record is not None, "Resolve the source record first."

verified_record = verify_landed_record(source_record)
landed_inspection = inspect_csv_gz_header(verified_record["landing_path"])

assert landed_inspection["column_count"] > 0
assert not landed_inspection["duplicate_columns"], "duplicate header names"
assert not landed_inspection["missing_columns"], "empty header names"
assert landed_inspection["schema_version"] == verified_record["schema_version"], (
    "schema drift: the landed header no longer matches the registered schema"
)
assert landed_inspection["byte_size"] == verified_record["byte_size"]

print(f"landed file     : {verified_record['landing_path']}")
print(f"hash verified   : {verified_record['file_hash_sha256']}")
print(f"byte size       : {verified_record['byte_size']}")
print(f"columns         : {landed_inspection['column_count']}")
print(f"fingerprint     : {landed_inspection['schema_fingerprint']}")
print("Landed object verified against the manifest; header valid, no drift.")

# Read the landed source file

In [0]:
# Read the immutable landed source object resolved from the manifest.
#
# The F05.2 input path is the manifest landing_path - never the old
# root Volume path and never the incoming staging copy.
# inferSchema=False keeps source values as strings in Bronze.
# Type cleaning and casting belong to Silver, not Bronze.

raw_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .option("multiLine", True)
    .option("escape", '"')
    .csv(source_record["landing_path"])
)

# Count input rows now so we can compare them with Bronze output later.
raw_row_count = raw_df.count()

print(f"Input path: {source_record['landing_path']}")
print(f"Raw rows read: {raw_row_count}")
display(raw_df.limit(5))

# Add provenance metadata

In [0]:
# Build provenance metadata for this specific ingestion run.
#
# These fields make every Bronze row traceable to its source file,
# city, snapshot date, pipeline execution, and - since F05.2 - the
# manifest-backed source-file identity and content hash. _raw_path is
# the immutable landing object this run actually read.

provenance = build_provenance_columns(
    source=SOURCE,
    city=CITY,
    snapshot_date=SNAPSHOT_DATE,
    raw_path=source_record["landing_path"],
    run_id=run_id,
    source_file_id=source_record["source_file_id"],
    file_hash_sha256=source_record["file_hash_sha256"],
)

# Add metadata columns to every raw Airbnb record.
# The underscore prefix separates pipeline metadata from original Airbnb fields.
bronze_df = (
    raw_df
    .withColumn("_source", F.lit(provenance["source"]))
    .withColumn("_city", F.lit(provenance["city"]))
    .withColumn("_snapshot_date", F.lit(provenance["snapshot_date"]))
    .withColumn("_raw_path", F.lit(provenance["raw_path"]))
    .withColumn("_run_id", F.lit(provenance["run_id"]))
    .withColumn("_ingested_at_utc", F.lit(provenance["ingested_at_utc"]))
    .withColumn("_source_file_id", F.lit(provenance["source_file_id"]))
    .withColumn("_file_hash_sha256", F.lit(provenance["file_hash_sha256"]))
)

display(bronze_df.limit(5))

# Replace only the snapshot slice

In [0]:
# Replace only this city/snapshot slice (F05.2).
#
# write_snapshot is the single tested write path: one atomic Delta
# overwrite with replaceWhere, so the slice is never observed
# half-written between a delete and an append. merge_schema=True lets
# the table pick up new provenance columns (e.g. _source_file_id added
# in F05.2) while every other city/snapshot slice remains untouched.
# The previous DELETE + APPEND pair was removed so exactly one write
# implementation exists in this pipeline.

write_snapshot(
    bronze_df,
    BRONZE_TABLE,
    CITY,
    SNAPSHOT_DATE,
    "_city",
    "_snapshot_date",
    merge_schema=True,
)

# Count what was actually written for verification.
bronze_row_count = (
    spark.table(BRONZE_TABLE)
    .filter(
        (F.col("_city") == CITY)
        & (F.col("_snapshot_date") == SNAPSHOT_DATE)
    )
    .count()
)

print(f"Target table: {BRONZE_TABLE}")
print(f"Bronze rows written for {CITY} / {SNAPSHOT_DATE}: {bronze_row_count}")

# Record success and verify

In [0]:
# Record completion time and mark this run successful.
completed_at_utc = datetime.now(UTC).isoformat()

spark.sql(f"""
    UPDATE {CONTROL_TABLE}
    SET status = 'success',
        row_count = {bronze_row_count},
        completed_at_utc = '{completed_at_utc}',
        error_message = NULL
    WHERE run_id = '{run_id}'
""")

# Reconcile for the EXACT source identity (F05.2): every row written
# for this source file must come from one run and match the parsed
# source row count - not merely the whole city/snapshot slice.
verification_df = (
    spark.table(BRONZE_TABLE)
    .filter(F.col("_source_file_id") == source_record["source_file_id"])
    .agg(
        F.count("*").alias("bronze_row_count"),
        F.countDistinct("_run_id").alias("distinct_run_ids"),
    )
)

display(verification_df)

identity_counts = verification_df.collect()[0]
assert identity_counts["distinct_run_ids"] == 1, (
    f"Expected one run identity, found {identity_counts['distinct_run_ids']}"
)

# The Bronze row count must match the raw input row count, both for the
# slice and for the exact source-file identity.
assert bronze_row_count == raw_row_count, (
    f"Row-count mismatch: raw={raw_row_count}, bronze={bronze_row_count}"
)
assert identity_counts["bronze_row_count"] == raw_row_count, (
    f"Source-identity mismatch: raw={raw_row_count}, "
    f"bronze for this source_file_id={identity_counts['bronze_row_count']}"
)

print("Bronze ingestion verification passed.")

In [0]:
# Post-run summary of every loaded city/snapshot slice.
#
# The table name flows through the run mode, so a test run summarizes
# the isolated test table rather than the development table.
display(
    spark.sql(f"""
        SELECT
            _city,
            _snapshot_date,
            COUNT(*) AS row_count,
            COUNT(DISTINCT _run_id) AS run_count,
            COUNT(DISTINCT _source_file_id) AS source_file_count
        FROM {BRONZE_TABLE}
        GROUP BY _city, _snapshot_date
    """)
)